In [1]:
import os
import time
import glob
import io
import re
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException
from webdriver_manager.chrome import ChromeDriverManager

class ScraperSIAP:
    def __init__(self, download_dir="temp_downloads"):
        self.base_url = "https://nube.agricultura.gob.mx/avance_agricola/"
        self.download_dir = os.path.abspath(download_dir)
        
        if not os.path.exists(self.download_dir):
            os.makedirs(self.download_dir)
            
        options = webdriver.ChromeOptions()
        prefs = {
            "download.default_directory": self.download_dir,
            "download.prompt_for_download": False,
            "download.directory_upgrade": True,
            "safebrowsing.enabled": True,
            "profile.default_content_settings.popups": 0
        }
        options.add_experimental_option("prefs", prefs)
        options.page_load_strategy = 'normal'
        
        self.driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        self.wait = WebDriverWait(self.driver, 20)

    def esperar_desbloqueo_ui(self):
        """Espera a que desaparezca la cortina de carga 'blockOverlay'."""
        try:
            WebDriverWait(self.driver, 5).until(
                EC.invisibility_of_element_located((By.CLASS_NAME, "blockOverlay"))
            )
        except TimeoutException:
            pass

    def click_robusto(self, elemento):
        """Clic que maneja intercepciones y esperas."""
        try:
            self.esperar_desbloqueo_ui()
            elemento.click()
        except ElementClickInterceptedException:
            self.driver.execute_script("arguments[0].click();", elemento)

    def esperar_elemento(self, locator_type, locator_value, condicion="clickable"):
        try:
            self.esperar_desbloqueo_ui()
            if condicion == "clickable":
                return self.wait.until(EC.element_to_be_clickable((locator_type, locator_value)))
            elif condicion == "visible":
                return self.wait.until(EC.visibility_of_element_located((locator_type, locator_value)))
        except TimeoutException:
            return None

    def iniciar_navegador(self):
        print("🌍 Conectando al SIAP...")
        self.driver.get(self.base_url)
        
        btn_cultivo = self.esperar_elemento(By.ID, "tipo-cult", condicion="clickable")
        if btn_cultivo:
            self.click_robusto(btn_cultivo)
            if self.esperar_elemento(By.ID, "anioagric", condicion="visible"):
                print("✅ Menú inicial cargado.")
            else:
                raise Exception("Menú de años no apareció.")
        else:
            raise Exception("Botón inicial no encontrado.")

    def seleccionar_opcion(self, element_id, value):
        try:
            self.esperar_desbloqueo_ui()
            elem = self.esperar_elemento(By.ID, element_id, condicion="clickable")
            if not elem: return False
            
            select = Select(elem)
            options_text = [o.get_attribute("value") for o in select.options]
            
            if str(value) not in options_text:
                return False
                
            select.select_by_value(str(value))
            # Pequeña pausa para que el JS reconozca el cambio antes de seguir
            time.sleep(0.5) 
            return True
        except Exception:
            return False

    def procesar_archivo_final(self, ruta_archivo, meta_info):
        """
        Escáner Inverso con BeautifulSoup.
        Extrae Año Agrícola del HTML e inyecta el resto de metadatos.
        """
        try:
            with open(ruta_archivo, 'rb') as f:
                content = f.read()
            
            html_content = ""
            for encoding in ['utf-8', 'latin-1', 'cp1252']:
                try:
                    html_content = content.decode(encoding)
                    break
                except: continue
                
            if not html_content: return pd.DataFrame()
            soup = BeautifulSoup(html_content, 'html.parser')
        except Exception as e:
            print(f"❌ Error lectura: {e}")
            return pd.DataFrame()

        datos_totales = []
        tablas = soup.find_all('table')

        for tabla in tablas:
            anio_agricola = "SIN_DATO"
            
            # --- ESCÁNER INVERSO (Solo para Año Agrícola) ---
            texto_reciente = "" 
            contador_pasos = 0
            
            for elem in tabla.previous_elements:
                contador_pasos += 1
                if contador_pasos > 200: break 
                
                if isinstance(elem, str):
                    texto_actual = elem.strip()
                    if not texto_actual: continue
                    
                    contexto_unido = texto_actual + " " + texto_reciente
                    
                    if "año agrícola" in contexto_unido.lower():
                        match = re.search(r'(\d{4})', contexto_unido)
                        if match:
                            anio_agricola = match.group(1)
                            break 
                    
                    texto_reciente = texto_actual

            # --- PROCESAR TABLA ---
            try:
                df = pd.read_html(io.StringIO(str(tabla)), header=None)[0]
                df = df.astype(str)
            except ValueError:
                continue

            # --- EXTRACCIÓN DE DATOS ---
            for idx, row in df.iterrows():
                cells = [str(c).strip() for c in row.values if str(c).lower() != 'nan']
                row_str = " ".join(cells).lower()
                
                if "total" in row_str or "sembrada" in row_str: continue
                
                col_cultivo = -1
                numeros = []
                
                for col_idx, cell in enumerate(row.values):
                    val = str(cell).strip().replace(',', '')
                    es_numero = False
                    try:
                        float(val); es_numero = True
                    except: pass
                    
                    if not es_numero and len(val) > 2 and col_cultivo == -1:
                        if val.lower() not in ["superficie", "producción", "rendimiento", "volumen"]:
                            col_cultivo = col_idx
                    
                    if es_numero and col_cultivo != -1 and col_idx > col_cultivo:
                        numeros.append(float(val))
                
                if col_cultivo != -1 and len(numeros) >= 1:
                    datos_totales.append({
                        'Año Reporte': meta_info['year'],
                        'Mes Reporte': meta_info['month'],
                        'Estado': meta_info['state_name'],
                        'Ciclo': meta_info.get('ciclo_name'),
                        'Modalidad': meta_info.get('modalidad_name'),
                        'Año Agrícola': anio_agricola,
                        'Cultivo': row.values[col_cultivo].strip(),
                        'Sembrada': numeros[0],
                        'Cosechada': numeros[1] if len(numeros) > 1 else 0.0,
                        'Siniestrada': numeros[2] if len(numeros) > 2 else 0.0,
                        'Produccion': numeros[3] if len(numeros) > 3 else 0.0,
                        'Rendimiento': numeros[4] if len(numeros) > 4 else 0.0
                    })

        return pd.DataFrame(datos_totales)

    def descargar_y_procesar(self, meta_info):
        # 1. PASO NUEVO: CLICK EN "CONSULTAR"
        # Esto actualiza la tabla en pantalla con los filtros seleccionados
        btn_consultar = self.esperar_elemento(By.ID, "Consultar", condicion="clickable")
        if btn_consultar:
            self.click_robusto(btn_consultar)
            # Esperamos a que la petición termine y la pantalla se desbloquee
            self.esperar_desbloqueo_ui() 
            time.sleep(2) # Pausa de seguridad para renderizado de tabla
        else:
            print("         ⚠️ No se encontró botón Consultar.")
            return None

        # 2. Limpiar descargas previas
        files = glob.glob(os.path.join(self.download_dir, "*"))
        for f in files: 
            try: os.remove(f)
            except: pass

        # 3. Generar Excel (Ahora sí tendrá los datos correctos)
        btn_generar = self.esperar_elemento(By.ID, "Excel", condicion="clickable")
        if not btn_generar: return None
        self.click_robusto(btn_generar)

        # 4. Esperar descarga
        tiempo_max = 30
        start = time.time()
        archivo_descargado = None
        
        while time.time() - start < tiempo_max:
            archivos = [f for f in glob.glob(os.path.join(self.download_dir, "*")) 
                       if not f.endswith('.crdownload') and not f.endswith('.tmp')]
            if archivos:
                archivo_descargado = max(archivos, key=os.path.getctime)
                break
            time.sleep(1)
            
        if not archivo_descargado:
            print("         ⚠️ Timeout descarga.")
            return None
            
        print(f"         📄 Procesando: {os.path.basename(archivo_descargado)}")

        # 5. Procesar
        df_result = self.procesar_archivo_final(archivo_descargado, meta_info)
        
        # 6. Limpieza
        try: os.remove(archivo_descargado)
        except: pass
            
        return df_result

    def cerrar(self):
        try: self.driver.quit()
        except: pass

def main():
    bot = ScraperSIAP()
    all_data = []
    
    # === CONFIGURACIÓN ===
    ANIOS = [2024] 
    MESES = { 1: "Enero", 2: "Febrero" } 
    ENTIDADES = [0] # 0 = Nacional
    
    try:
        bot.iniciar_navegador()
        print("⚙️ Configurando filtros base...")
        bot.seleccionar_opcion("cicloProd", "4") # Año Agrícola
        bot.seleccionar_opcion("modalidad", "3") # Riego + Temporal
        
        for year in ANIOS:
            print(f"\n📅 AÑO REPORTE: {year}")
            if not bot.seleccionar_opcion("anioagric", year): continue
            
            for mes_num, mes_nombre in MESES.items():
                print(f"   🗓️ {mes_nombre}")
                if not bot.seleccionar_opcion("mesagric", mes_num): continue
                
                for ent_id in ENTIDADES:
                    bot.seleccionar_opcion("entidad", ent_id)
                    bot.seleccionar_opcion("cultivo", "0") 
                    
                    meta = {
                        'year': year, 
                        'month': mes_nombre,
                        'state_name': "Nacional" if ent_id == 0 else f"Entidad {ent_id}",
                        'ciclo_name': "Año Agrícola",
                        'modalidad_name': "Riego + Temporal"
                    }
                    
                    # Ahora esto incluye el click en "Consultar"
                    df_chunk = bot.descargar_y_procesar(meta)
                    
                    if df_chunk is not None and not df_chunk.empty:
                        all_data.append(df_chunk)
                        print(f"         ✅ ÉXITO: {len(df_chunk)} filas.")
                    else:
                        print(f"         ⚠️ Sin datos.")

    except KeyboardInterrupt:
        print("\n🛑 Stop usuario.")
    except Exception as e:
        print(f"\n❌ Error General: {e}")
    finally:
        bot.cerrar()
        if all_data:
            final_df = pd.concat(all_data, ignore_index=True)
            output_file = "SIAP_Final_Validado.csv"
            final_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            print(f"\n🚀 PROCESO TERMINADO. Archivo: {output_file}")
            print(final_df.head())
        else:
            print("\n⚠️ No se generó archivo final.")

if __name__ == "__main__":
    main()

🌍 Conectando al SIAP...
✅ Menú inicial cargado.
⚙️ Configurando filtros base...

🛑 Stop usuario.

⚠️ No se generó archivo final.
